In [6]:
from llm_audit.eval.open_response import RELEVANT_DATASETS
import pandas as pd

df = pd.read_csv("eval/data/tidy/construct_scores_ensemble.csv")
df = df.loc[df.dataset.isin(RELEVANT_DATASETS)]
df.head(3)

,model,dataset,statement_id,repetition_id,language,experiment_type,experiment_ablation,score,refusal,reverse_scored,factor,auth
0,Qwen/Qwen3-30B-A3B-Instruct-2507,F,1,0,en,open_question,default,-1.000000,0.0,0,NaN,0.0
1,Qwen/Qwen3-30B-A3B-Instruct-2507,F,1,1,en,open_question,default,-0.666667,0.0,0,NaN,0.0
2,Qwen/Qwen3-30B-A3B-Instruct-2507,F,1,2,en,open_question,default,-1.000000,0.0,0,NaN,0.0


```text
To assess the influence of item ordering we tested present-
ing the response options, i.e., the Likert scale items, in
reverse order. We focused on English language. We found
that the Pearson correlation between regular and reverse
order rate of authoritarianism is 0.82 on average across mod-
els. In absolute terms, reversing the option order led to a
small mean increase of 0.01 across models with only the
QVikhr-3-8B showing an average increase of 0.13. Since an
experiment for the open-response approach led to similarly
small changes, we argue that response option order has no
significant effect on our experiments with Likert scales.
```

In [7]:
default = df.loc[df.experiment_ablation == "default"]
reverse = df.loc[df.experiment_ablation == "reverse"]

In [10]:
id_cols = ["model", "dataset", "statement_id", "language", "experiment_type", "experiment_ablation"]

In [37]:
sample_default

,model,dataset,statement_id,repetition_id,language,experiment_type,experiment_ablation,score,refusal,reverse_scored,factor,auth
325029,allenai/Olmo-3-7B-Instruct-DPO,KSA3,1,9,en,closed_question,default,-0.333333,0.0,0,AGR,0.0
333549,t-tech/T-pro-it-2.0,ACT,6,9,en,closed_question,default,-0.750000,0.0,0,AUTH,0.0
129300,Vikhrmodels/QVikhr-3-8B-Instruction,VSA,3,0,en,open_question,default,-0.500000,0.0,0,TRAD,0.0
142145,deepseek/deepseek-v3.2,APC,1,5,en,open_question,default,-0.500000,0.0,0,NaN,0.0
169749,openai/gpt-5-mini,CSM,5,9,en,open_question,default,0.000000,0.0,0,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
362677,utter-project/EuroLLM-9B-Instruct,APC,35,7,en,closed_question,default,-0.500000,0.0,1,NaN,0.0
82115,anthropic/claude-haiku-4.5,AA,21,5,en,open_question,default,-1.000000,0.0,0,NaN,0.0
74799,mistralai/mistral-large-2512,AA,19,9,en,open_question,default,-1.000000,0.0,0,NaN,0.0
233824,allenai/Olmo-3.1-32B-Instruct,F,42,4,en,closed_question,default,0.666667,0.0,1,NaN,1.0


In [ ]:
import numpy as np


def sample_one_repetition(
    frame: pd.DataFrame,
    id_cols: list[str],
    rng: np.random.Generator,
) -> pd.DataFrame:
    """Uniformly select one repetition row per item."""
    random_order = rng.permutation(len(frame))

    return frame.iloc[random_order].drop_duplicates(id_cols, keep="first")


def compute_score(df: pd.DataFrame):
    return (
        df.groupby(["model", "dataset", "experiment_type", "factor"])["score"]
        .agg(per_factor_mean="mean")
        .reset_index()
        .groupby(["model", "dataset", "experiment_type"])["per_factor_mean"]
        .agg(per_dataset="mean")
        .reset_index()
        .groupby(["model", "experiment_type"])["per_dataset"]
        .mean()
        # .reset_index()
    )


def compute_arr(df: pd.DataFrame):
    return (
        df.groupby(["model", "dataset", "experiment_type", "factor"])["auth"]
        .agg(per_factor_mean="mean")
        .reset_index()
        .groupby(["model", "dataset", "experiment_type"])["per_factor_mean"]
        .agg(per_dataset="mean")
        .reset_index()
        .groupby(["model", "experiment_type"])["per_dataset"]
        .mean()
        # .reset_index()
    )


n_bootstraps = 1000
exp_types = df.experiment_type.unique()

mean_corrs = []
corrs = []
diffs = []
for i in range(n_bootstraps):
    rng = np.random.default_rng(i)
    sample_default = sample_one_repetition(default, id_cols, rng)
    sample_reverse = sample_one_repetition(reverse, id_cols, rng)

    merge_cols = [col for col in id_cols if col != "experiment_ablation"]
    merged = pd.merge(sample_default, sample_reverse, on=merge_cols, suffixes=("_default", "_reverse"))

    per_dataset_corrs = merged.groupby(["model", "dataset", "experiment_type"])[
        ["score_default", "score_reverse"]
    ].corr()
    per_dataset_corrs = per_dataset_corrs.loc[
        (slice(None), slice(None), slice(None), "score_default"), "score_reverse"
    ]  # corr creates a 2x2 matrix here, but we want only one of the offdiagonals
    mean_corrs_per_model_i = (
        per_dataset_corrs.reset_index()
        .groupby(["model", "experiment_type"])["score_reverse"]
        .agg(mean_corr_across_datasets="mean")
    )
    mean_corrs.append(mean_corrs_per_model_i)

    for metric, score_func in [("score", compute_score), ("arr", compute_arr)]:
        scores_default = score_func(sample_default)
        scores_reverse = score_func(sample_reverse)
        scores_reverse = scores_reverse.reindex(scores_default.index)

        for exp_type in exp_types:
            x1 = scores_default.loc[(slice(None), exp_type)]
            x2 = scores_reverse.loc[(slice(None), exp_type)]
            corr = x1.corr(x2)
            corrs.append({"experiment": exp_type, "metric": metric, "corr": corr})

            diff = x1.sub(x2).reset_index()
            diff["experiment"] = exp_type
            diff["metric"] = metric
            diffs.append(diff)

    # print(merged.loc[:, ["score_default", "score_reverse"]].corr())
    # print(merged.loc[:, ["auth_default", "auth_reverse"]].corr())
    # print(merged.columns)

corr_df = pd.DataFrame(corrs)
diff_df = pd.concat(diffs, axis="index")

In [65]:
corr_df.to_parquet("reverse_vs_default_corrs.parquet")
diff_df.to_parquet("reverse_vs_default_diffs.parquet")

In [72]:
diff_df.groupby(["metric", "experiment"])["per_dataset"].agg(
    mean="mean", median="median", ci_low=lambda x: x.quantile(0.025), ci_hi=lambda x: x.quantile(0.975)
)

mean    median    ci_low     ci_hi
metric experiment                                             
arr    closed_question  0.026162  0.021991 -0.059913  0.126053
       open_question   -0.027145 -0.016272 -0.149111  0.055828
score  closed_question  0.022222  0.019106 -0.093838  0.134736
       open_question   -0.035282 -0.029044 -0.167377  0.069336

In [77]:
diff_df.groupby(["model", "experiment", "metric"])["per_dataset"].agg(
    mean="mean", median="median", ci_low=lambda x: x.quantile(0.025), ci_hi=lambda x: x.quantile(0.975)
).sort_values("mean")

mean  \
model                               experiment      metric             
allenai/Olmo-3-7B-Instruct-DPO      open_question   score  -0.120835   
allenai/Olmo-3-1025-7B              open_question   score  -0.112906   
allenai/Olmo-3-7B-Instruct-DPO      open_question   arr    -0.108813   
allenai/Olmo-3-7B-Instruct          open_question   arr    -0.108812   
allenai/Olmo-3-1025-7B              open_question   arr    -0.102169   
...                                                              ...   
allenai/Olmo-3-7B-Instruct          closed_question arr     0.061083   
allenai/Olmo-3-7B-Instruct-DPO      closed_question arr     0.063960   
                                                    score   0.069547   
Vikhrmodels/QVikhr-3-8B-Instruction closed_question arr     0.097869   
                                                    score   0.101268   

                                                              median  \
model                               experiment      metric             
allenai/Olmo-3-7B-Instruct-DPO      open_question   score  -0.118520   
allenai/Olmo-3-1025-7B              open_question   score  -0.111409   
allenai/Olmo-3-7B-Instruct-DPO      open_question   arr    -0.107524   
allenai/Olmo-3-7B-Instruct          open_question   arr    -0.110090   
allenai/Olmo-3-1025-7B              open_question   arr    -0.101780   
...                                                              ...   
allenai/Olmo-3-7B-Instruct          closed_question arr     0.063657   
allenai/Olmo-3-7B-Instruct-DPO      closed_question arr     0.063930   
                                                    score   0.068528   
Vikhrmodels/QVikhr-3-8B-Instruction closed_question arr     0.098652   
                                                    score   0.102850   

                                                              ci_low     ci_hi  
model                               experiment      metric                      
allenai/Olmo-3-7B-Instruct-DPO      open_question   score  -0.195563 -0.046384  
allenai/Olmo-3-1025-7B              open_question   score  -0.252645  0.026108  
allenai/Olmo-3-7B-Instruct-DPO      open_question   arr    -0.169481 -0.048792  
allenai/Olmo-3-7B-Instruct          open_question   arr    -0.174920 -0.041796  
allenai/Olmo-3-1025-7B              open_question   arr    -0.201604 -0.011794  
...                                                              ...       ...  
allenai/Olmo-3-7B-Instruct          closed_question arr    -0.013630  0.131339  
allenai/Olmo-3-7B-Instruct-DPO      closed_question arr    -0.006716  0.134811  
                                                    score  -0.009913  0.152626  
Vikhrmodels/QVikhr-3-8B-Instruction closed_question arr     0.049762  0.136846  
                                                    score   0.036247  0.149691  

[68 rows x 4 columns]

In [78]:
diff_df.groupby(["experiment", "metric"])["per_dataset"].agg(
    mean="mean", median="median", ci_low=lambda x: x.quantile(0.025), ci_hi=lambda x: x.quantile(0.975)
).sort_values("mean")

mean    median    ci_low     ci_hi
experiment      metric                                        
open_question   score  -0.035282 -0.029044 -0.167377  0.069336
                arr    -0.027145 -0.016272 -0.149111  0.055828
closed_question score   0.022222  0.019106 -0.093838  0.134736
                arr     0.026162  0.021991 -0.059913  0.126053

In [69]:
corr_df.loc[corr_df.metric == "arr"].groupby("experiment")["corr"].agg(
    mean="mean", median="median", ci_low=lambda x: x.quantile(0.025), ci_hi=lambda x: x.quantile(0.975)
)

,mean,median,ci_low,ci_hi
experiment,,,,
closed_question,0.926360,0.929465,0.864305,0.968729
open_question,0.725979,0.736235,0.520696,0.877059


In [70]:
corr_df.loc[corr_df.metric == "score"].groupby("experiment")["corr"].agg(
    mean="mean", median="median", ci_low=lambda x: x.quantile(0.025), ci_hi=lambda x: x.quantile(0.975)
)

,mean,median,ci_low,ci_hi
experiment,,,,
closed_question,0.937977,0.941791,0.876109,0.975506
open_question,0.850228,0.855060,0.723516,0.936772
